In [ ]:
import torch.multiprocessing as mp
try:
    mp.set_start_method("spawn", force=True)  # Windows/Jupyter safe
except RuntimeError:
    pass  # already set is fine


In [ ]:
import os
import time
import csv
from pathlib import Path

import torch
import torchvision
import torchvision.transforms as T
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import DataLoader
from PIL import Image
import numpy as np
from sklearn.metrics import precision_score, recall_score, confusion_matrix, precision_recall_curve, f1_score
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torchvision.ops import box_iou
import xml.etree.ElementTree as ET
from tqdm import tqdm
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d as uf
import pickle
import pandas as pd

# ============================================================================
# CONFIGURATION SECTION - EASILY MODIFIABLE
# ============================================================================

# Class Configuration - Modify this list to add/remove classes
CLASS_NAMES = [
    "fall",
    "no_fall"
]

# Create class mapping automatically (1-indexed, 0 is background)
CLASS_MAP = {name.lower().replace(" ", "_"): idx + 1 for idx, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES) + 1  # +1 for background class

# Training Configuration
VISUAL_DEBUG = False
SCORE_THRESHOLD = 0.5
IOU_THRESHOLD = 0.5
INPUT_SIZE = 640

# Hyperparameters
NUM_EPOCHS = 100
LEARNING_RATE = 0.1
BATCH_SIZE = 16
VAL_BATCH_SIZE = 1
MOMENTUM = 0.937
WEIGHT_DECAY = 0.0005
PATIENCE = 5
WARMUP_EPOCH = 3

# Dataset Configuration
DATASET_BASE = 'dataset_paper_new'

print(f"Configured for {len(CLASS_NAMES)} classes:")
for name, idx in CLASS_MAP.items():
    print(f"  {idx}: {name}")

# ============================================================================
# DATASET AND TRANSFORMS
# ============================================================================

def get_transform():
    return T.Compose([T.Resize((INPUT_SIZE, INPUT_SIZE)), T.ToTensor()])

class VOCLikeDataset(torch.utils.data.Dataset):
    def __init__(self, images_dir, labels_dir, transforms=None, class_map=None):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.transforms = transforms
        self.class_map = class_map or CLASS_MAP
        self.files = sorted([f for f in os.listdir(images_dir) if f.endswith((".jpg",".png"))])
        
    def __len__(self): 
        return len(self.files)
        
    def __getitem__(self, idx):
        img_file = self.files[idx]
        img = Image.open(os.path.join(self.images_dir, img_file)).convert("RGB")
        xml_path = os.path.join(self.labels_dir, img_file.rsplit('.',1)[0] + '.xml')
        
        tree = ET.parse(xml_path)
        boxes, labels = [], []
        
        for obj in tree.getroot().findall('object'):
            cls = obj.find('name').text.lower().replace(" ", "_")
            if cls not in self.class_map:
                print(f"Warning: Unknown class '{cls}' found in {xml_path}")
                continue
                
            labels.append(self.class_map[cls])
            b = obj.find('bndbox')
            boxes.append([
                float(b.find('xmin').text), 
                float(b.find('ymin').text),
                float(b.find('xmax').text), 
                float(b.find('ymax').text)
            ])
        
        if self.transforms:
            img = self.transforms(img)
            
        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.int64)
        }
        return img, target

# ============================================================================
# VALIDATION FUNCTION
# ============================================================================

def validate(model, loader, device):
    """
    Runs model on loader, returns:
      - losses dict with keys 'box','cls','dfl'
      - metrics dict with keys 'precision','recall','mAP50','mAP50-95'
      - lists all_t, all_p, all_s for PR curves
    """
    model.eval()
    mp50 = MeanAveragePrecision(iou_thresholds=[0.5]).to(device)
    mp_all = MeanAveragePrecision().to(device)
    sum_b = sum_c = sum_d = 0.0
    count = 0
    all_t, all_p, all_s = [], [], []

    def match_predictions(pred_boxes, pred_labels, pred_scores, gt_boxes, gt_labels, iou_thresh=0.5):
        matches = []
        if len(pred_boxes) == 0:
            # No predictions, all ground truth are false negatives
            for j in range(len(gt_boxes)):
                matches.append((0, gt_labels[j].item(), 0.0))
            return matches
            
        if len(gt_boxes) == 0:
            # No ground truth, all predictions are false positives
            for i in range(len(pred_boxes)):
                matches.append((pred_labels[i].item(), 0, pred_scores[i].item()))
            return matches
            
        ious = box_iou(pred_boxes, gt_boxes)
        gt_used = set()
        
        for i in range(len(pred_boxes)):
            score = pred_scores[i].item()
            label = pred_labels[i].item()
            
            if ious.numel() > 0:
                max_iou, gt_idx = ious[i].max(0)
                if max_iou >= iou_thresh and gt_idx.item() not in gt_used:
                    matches.append((label, gt_labels[gt_idx].item(), score))
                    gt_used.add(gt_idx.item())
                    continue
            matches.append((label, 0, score))
            
        # Add unmatched ground truth as false negatives
        for j in range(len(gt_boxes)):
            if j not in gt_used:
                matches.append((0, gt_labels[j].item(), 0.0))
        return matches

    with torch.no_grad():
        for imgs, targets in loader:
            imgs = [img.to(device) for img in imgs]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            # Get predictions
            outputs = model(imgs)
            
            # Get training losses
            model.train()
            loss_dict = model(imgs, targets)
            model.eval()
            
            # Convert tensors to Python floats immediately
            sum_b += float(loss_dict['loss_box_reg'].item())
            sum_c += float(loss_dict['loss_classifier'].item())
            sum_d += float(loss_dict.get('loss_objectness', 0.0)) + float(loss_dict.get('loss_rpn_box_reg', 0.0))
            count += 1
            
            # Process predictions for metrics
            for out, tgt in zip(outputs, targets):
                keep = out['scores'] > SCORE_THRESHOLD
                pred = {
                    'boxes': out['boxes'][keep], 
                    'scores': out['scores'][keep], 
                    'labels': out['labels'][keep]
                }
                gt = {'boxes': tgt['boxes'], 'labels': tgt['labels']}
                
                mp50.update([pred], [gt])
                mp_all.update([pred], [gt])
                
                pb, pl, ps = pred['boxes'].cpu(), pred['labels'].cpu(), pred['scores'].cpu()
                gb, gl = gt['boxes'].cpu(), gt['labels'].cpu()
                
                for p_label, g_label, p_score in match_predictions(pb, pl, ps, gb, gl, iou_thresh=IOU_THRESHOLD):
                    all_p.append(p_label)
                    all_t.append(g_label)
                    all_s.append(p_score)

    res50 = float(mp50.compute()['map'].item())
    res_all = float(mp_all.compute()['map'].item())
    
    valid_labels = list(range(1, NUM_CLASSES))
    precision = precision_score(all_t, all_p, labels=valid_labels, average='weighted', zero_division=0)
    recall = recall_score(all_t, all_p, labels=valid_labels, average='weighted', zero_division=0)
    
    losses = {'box': sum_b/count, 'cls': sum_c/count, 'dfl': sum_d/count}
    metrics = {'precision': precision, 'recall': recall, 'mAP50': res50, 'mAP50-95': res_all}
    
    return losses, metrics, all_t, all_p, all_s

# ============================================================================
# PLOTTING FUNCTIONS
# ============================================================================

def create_confusion_matrix(y_true, y_pred, save_path, normalize=False):
    """Create confusion matrix plot"""
    labels = list(range(1, NUM_CLASSES))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    
    if normalize:
        cm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        cm = np.nan_to_num(cm)  # Handle division by zero
        title = 'Confusion Matrix (Normalized)'
        fmt = '.2f'
        vmax = 1.0
    else:
        title = 'Confusion Matrix'
        fmt = 'd'
        vmax = None
    
    # Calculate figure size based on number of classes
    fig_size = max(8, len(CLASS_NAMES) * 0.8)
    fig, ax = plt.subplots(figsize=(fig_size, fig_size))
    
    im = ax.imshow(cm, cmap='Blues', vmax=vmax)
    
    # Add text annotations
    for (i, j), v in np.ndenumerate(cm):
        color = 'white' if (normalize and v > 0.5) or (not normalize and v > cm.max()/2) else 'black'
        if normalize:
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', color=color, fontsize=8)
        else:
            ax.text(j, i, f'{v}', ha='center', va='center', color=color, fontsize=8)
    
    ax.set_xticks(range(len(CLASS_NAMES)), labels=CLASS_NAMES, rotation=45, ha='right')
    ax.set_yticks(range(len(CLASS_NAMES)), labels=CLASS_NAMES)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title)
    
    plt.colorbar(im)
    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

def create_confidence_curves(all_t, all_p, all_s, plot_dir):
    """Create F1, Precision, and Recall confidence curves"""
    labels = list(range(1, NUM_CLASSES))
    thr = np.linspace(0, 1, 200)
    
    # F1 Curve
    f1_curves = {}
    f1_all = []
    
    yta, ypa, ysa = np.array(all_t), np.array(all_p), np.array(all_s)
    
    for t in thr:
        yp = np.where(ysa >= t, ypa, 0)
        f1_all.append(f1_score(yta, yp, labels=labels, average='weighted', zero_division=0))
        
        # Calculate per-class F1
        for i, label in enumerate(labels):
            if label not in f1_curves:
                f1_curves[label] = []
            f1_curves[label].append(f1_score((yta == label).astype(int), (yp == label).astype(int), zero_division=0))
    
    # Plot F1 curve
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Plot top classes (limit to avoid clutter)
    top_classes = min(5, len(CLASS_NAMES))
    for i, (label, class_name) in enumerate(zip(labels[:top_classes], CLASS_NAMES[:top_classes])):
        ax.plot(thr, f1_curves[label], label=class_name, alpha=0.7)
    
    best = int(np.nanargmax(f1_all))
    ax.plot(thr, f1_all, linewidth=3, label=f'All Classes {f1_all[best]:.2f}@{thr[best]:.2f}', color='black')
    
    ax.set_xlabel('Confidence')
    ax.set_ylabel('F1')
    ax.set_title(f'F1-Confidence Curve (Top {top_classes} Classes)')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    fig.tight_layout()
    fig.savefig(plot_dir/'F1_curve.png', dpi=300, bbox_inches='tight')
    plt.close(fig)
    
    # Precision and Recall curves
    for metric_name, metric_func in [('Precision', precision_score), ('Recall', recall_score)]:
        fig, ax = plt.subplots(figsize=(12, 8))
        
        # Per-class curves
        for cls, class_name in zip(labels, CLASS_NAMES):
            mask = (yta == cls).astype(int)
            scores = np.where(ypa == cls, ysa, 0.0)
            
            if len(np.unique(mask)) > 1:  # Only plot if class exists
                prec_ci, rec_ci, thr_ci = precision_recall_curve(mask, scores)
                if metric_name == 'Precision':
                    y_vals = prec_ci[:-1]
                else:
                    y_vals = rec_ci[:-1]
                x_vals = thr_ci
                area = np.trapz(y_vals, x_vals) if len(x_vals) > 1 else 0
                ax.plot(x_vals, y_vals, label=f"{class_name} {area:.3f}", alpha=0.7)
        
        # All classes curve
        metric_all = []
        for t in thr:
            yp_thresh = np.where(ysa >= t, ypa, 0)
            metric_all.append(metric_func(yta, yp_thresh, labels=labels, average='weighted', zero_division=0))
        
        best_idx = int(np.nanargmax(metric_all))
        area_all = np.trapz(metric_all, thr)
        ax.plot(thr, metric_all, linewidth=3, color='black',
                label=f"All Classes {area_all:.3f} at {thr[best_idx]:.3f}")
        
        ax.set_xlabel('Confidence')
        ax.set_ylabel(metric_name)
        ax.set_title(f'{metric_name}–Confidence Curve')
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        fig.tight_layout()
        fig.savefig(plot_dir/f'{metric_name[0]}_curve.png', dpi=300, bbox_inches='tight')
        plt.close(fig)

def create_pr_curve(all_t, all_p, all_s, save_path):
    """Create Precision-Recall curve"""
    labels = list(range(1, NUM_CLASSES))
    yta, ypa, ysa = np.array(all_t), np.array(all_p), np.array(all_s)
    
    fig, ax = plt.subplots(figsize=(12, 8))
    
    ap_scores = []
    for cls, class_name in zip(labels, CLASS_NAMES):
        mask = (yta == cls).astype(int)
        scores = np.where(ypa == cls, ysa, 0)
        
        if len(np.unique(mask)) > 1:  # Only plot if class exists
            prec, rec, _ = precision_recall_curve(mask, scores)
            ap = np.trapz(prec, rec) if len(rec) > 1 else 0
            ap_scores.append(ap)
            ax.plot(rec, prec, label=f"{class_name} AP={ap:.3f}", alpha=0.7)
        else:
            ap_scores.append(0)
    
    # Overall mAP
    overall_map = np.mean(ap_scores)
    ax.plot([], [], linewidth=3, color='black', label=f'mAP = {overall_map:.3f}')
    
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title('Precision-Recall Curve')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

def create_label_distribution(dataset_base, save_path):
    """Create label distribution bar chart"""
    ann = []
    labels_path = Path(f"{dataset_base}/labels_voc/train")
    
    if not labels_path.exists():
        print(f"Warning: Labels path {labels_path} does not exist")
        return
    
    for xml_file in labels_path.glob('*.xml'):
        try:
            tree = ET.parse(xml_file)
            for obj in tree.getroot().findall('object'):
                class_name = obj.find('name').text
                ann.append(class_name)
        except Exception as e:
            print(f"Error reading {xml_file}: {e}")
    
    if not ann:
        print("No annotations found")
        return
    
    counts = pd.Series(ann).value_counts()
    
    fig, ax = plt.subplots(figsize=(12, 8))
    bars = counts.plot.bar(ax=ax, color='skyblue', edgecolor='black')
    ax.set_ylabel('Number of Instances')
    ax.set_xlabel('Class')
    ax.set_title('Label Distribution in Training Set')
    ax.tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for bar in bars.patches:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                f'{int(height)}', ha='center', va='bottom')
    
    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

def create_results_panel(csv_file, save_path):
    """Create training results panel"""
    if not os.path.exists(csv_file):
        print(f"Results CSV file {csv_file} not found")
        return
        
    df = pd.read_csv(csv_file)
    smooth = lambda x: uf(x.astype(float), size=5, mode='nearest')
    
    cols = [
        ('train/box_loss', 'Box Loss'),
        ('train/cls_loss', 'Cls Loss'),
        ('train/dfl_loss', 'DFL Loss'),
        ('metrics/precision(B)', 'Precision'),
        ('metrics/recall(B)', 'Recall'),
        ('metrics/mAP50(B)', 'mAP50'),
        ('metrics/mAP50-95(B)', 'mAP50-95'),
        ('val/box_loss', 'Val Box'),
        ('val/cls_loss', 'Val Cls'),
        ('val/dfl_loss', 'Val DFL')
    ]
    
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    axes = axes.flatten()
    
    for ax, (col, name) in zip(axes, cols):
        if col in df.columns:
            raw = df[col].astype(float)
            ax.plot(df['epoch'], raw, marker='o', label='raw', alpha=0.7, markersize=3)
            ax.plot(df['epoch'], smooth(raw), linestyle='--', label='smooth', linewidth=2)
            ax.set_title(name)
            ax.set_xlabel('Epoch')
            ax.legend()
            ax.grid(True, alpha=0.3)
        else:
            ax.text(0.5, 0.5, f'No data for\n{name}', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(name)
    
    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

# ============================================================================
# MAIN TRAINING FUNCTION
# ============================================================================

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Create datasets
    train_ds = VOCLikeDataset(f"{DATASET_BASE}/images/train", f"{DATASET_BASE}/labels_voc/train", get_transform())
    val_ds = VOCLikeDataset(f"{DATASET_BASE}/images/val", f"{DATASET_BASE}/labels_voc/val", get_transform())
    
    # Create data loaders
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
    val_loader = DataLoader(val_ds, batch_size=VAL_BATCH_SIZE, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))
    
    print(f"Training samples: {len(train_ds)}")
    print(f"Validation samples: {len(val_ds)}")
    
    # Create model
    model = fasterrcnn_resnet50_fpn(weights='DEFAULT')
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
    model.to(device)
    
    # Create optimizer and scheduler
    optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    
    # Prepare save directory and CSV
    save_dir = Path(f'runs_2/train/FRCNN_noD_Normal_lr.{LEARNING_RATE}')
    save_dir.mkdir(parents=True, exist_ok=True)
    
    csv_file = save_dir / 'results.csv'
    headers = [
        'epoch', 'time', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss',
        'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)',
        'val/box_loss', 'val/cls_loss', 'val/dfl_loss', 'lr/pg0', 'lr/pg1', 'lr/pg2'
    ]
    
    best_map = 0
    patience_counter = 0

    # Training loop
    with open(csv_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        
        for epoch in range(1, NUM_EPOCHS + 1):
            model.train()
            t0 = time.time()
        
            if epoch <= WARMUP_EPOCH:
                warmup_lr = LEARNING_RATE * (epoch / WARMUP_EPOCH)
                for param_group in optimizer.param_groups:
                    param_group['lr'] = warmup_lr
                    param_group['momentum'] = 0.8
            
            # Training phase
            sum_box = sum_cls = sum_dfl = 0.0
            train_pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS}')
            
            for imgs, targets in train_pbar:
                imgs = [img.to(device) for img in imgs]
                targets = [{k: v.to(device) for k, v in target.items()} for target in targets]
                
                loss_dict = model(imgs, targets)
                losses = sum(loss_dict.values())
                
                sum_box += float(loss_dict['loss_box_reg'].item())
                sum_cls += float(loss_dict['loss_classifier'].item())
                sum_dfl += float(loss_dict.get('loss_objectness', 0.0)) + float(loss_dict.get('loss_rpn_box_reg', 0.0))
                
                optimizer.zero_grad()
                losses.backward()
                optimizer.step()
                
                # Update progress bar
                train_pbar.set_postfix({
                    'box': f'{sum_box/(train_pbar.n+1):.3f}',
                    'cls': f'{sum_cls/(train_pbar.n+1):.3f}'
                })
            
            # Calculate average training losses
            train_box = sum_box / len(train_loader)
            train_cls = sum_cls / len(train_loader)
            train_dfl = sum_dfl / len(train_loader)
            
            epoch_time = time.time() - t0
            
            # Validation phase
            print("Running validation...")
            val_losses, metrics, all_t, all_p, all_s = validate(model, val_loader, device)
            
            # Learning rates
            lrs = [group['lr'] for group in optimizer.param_groups]
            pg0, pg1, pg2 = (lrs + [lrs[-1]] * 3)[:3]
            
            # Write results
            row = [
                epoch, epoch_time,
                train_box, train_cls, train_dfl,
                metrics['precision'], metrics['recall'], metrics['mAP50'], metrics['mAP50-95'],
                val_losses['box'], val_losses['cls'], val_losses['dfl'],
                pg0, pg1, pg2
            ]
            writer.writerow(row)
            
            # Print epoch summary
            print(f"Epoch {epoch}/{NUM_EPOCHS} - Time: {epoch_time:.1f}s")
            print(f"  Train - Box: {train_box:.4f}, Cls: {train_cls:.4f}, DFL: {train_dfl:.4f}")
            print(f"  Val   - Box: {val_losses['box']:.4f}, Cls: {val_losses['cls']:.4f}, DFL: {val_losses['dfl']:.4f}")
            print(f"  Metrics - Precision: {metrics['precision']:.4f}, Recall: {metrics['recall']:.4f}")
            print(f"  mAP50: {metrics['mAP50']:.4f}, mAP50-95: {metrics['mAP50-95']:.4f}")
            print(f"  LR: {lrs[0]:.6f}")
            print("-" * 60)
            
            scheduler.step()
            
            # Early stopping check
            if metrics['mAP50'] > best_map:
                best_map = metrics['mAP50']
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter > PATIENCE:
                print(f"Early stopping at epoch {epoch} due to no improvement in mAP50")
                break
    
    # Final validation to collect predictions for plotting
    _, _, all_t, all_p, all_s = validate(model, val_loader, device)
    
    # Save predictions and model
    with open(save_dir / 'preds.pkl', 'wb') as f:
        pickle.dump((all_t, all_p, all_s), f)
    
    model_path = save_dir / 'model_final.pt'
    torch.save(model.state_dict(), model_path)
    print(f"Model weights saved to {model_path}")
    
    # Create plots
    print("Creating visualization plots...")
    plot_dir = save_dir / 'plots'
    plot_dir.mkdir(exist_ok=True)
    
    # 1. Confusion matrices
    create_confusion_matrix(all_t, all_p, plot_dir / 'confusion_matrix.png', normalize=False)
    create_confusion_matrix(all_t, all_p, plot_dir / 'confusion_matrix_normalized.png', normalize=True)
    
    # 2. Confidence curves (F1, Precision, Recall)
    create_confidence_curves(all_t, all_p, all_s, plot_dir)
    
    # 3. PR curve
    create_pr_curve(all_t, all_p, all_s, plot_dir / 'PR_curve.png')
    
    # 4. Label distribution
    create_label_distribution(DATASET_BASE, plot_dir / 'labels.png')
    
    # 5. Training results panel
    create_results_panel(csv_file, plot_dir / 'results.png')
    
    print(f"All plots saved under {plot_dir}")
    print(f"Training completed! Results saved in {save_dir}")

if __name__ == '__main__':
    main()


In [ ]:
import os
import time
import csv
from pathlib import Path

import torch
import torchvision
import torchvision.transforms as T
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import DataLoader
from PIL import Image
import numpy as np
from sklearn.metrics import precision_score, recall_score, confusion_matrix, precision_recall_curve, f1_score
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torchvision.ops import box_iou
import xml.etree.ElementTree as ET
from tqdm import tqdm
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d as uf
import pickle
import pandas as pd

# ============================================================================
# CONFIGURATION SECTION - EASILY MODIFIABLE
# ============================================================================

# Class Configuration - Modify this list to add/remove classes
CLASS_NAMES = [
    "fall",
    "no_fall"
]

# Create class mapping automatically (1-indexed, 0 is background)
CLASS_MAP = {name.lower().replace(" ", "_"): idx + 1 for idx, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES) + 1  # +1 for background class

# Training Configuration
VISUAL_DEBUG = False
SCORE_THRESHOLD = 0.5
IOU_THRESHOLD = 0.5
INPUT_SIZE = 640

# Hyperparameters
NUM_EPOCHS = 100
LEARNING_RATE = 0.1
BATCH_SIZE = 16
VAL_BATCH_SIZE = 1
MOMENTUM = 0.937
WEIGHT_DECAY = 0.0005
PATIENCE = 5
WARMUP_EPOCH = 3

# Dataset Configuration
DATASET_BASE = 'new_dataset'

print(f"Configured for {len(CLASS_NAMES)} classes:")
for name, idx in CLASS_MAP.items():
    print(f"  {idx}: {name}")

# ============================================================================
# DATASET AND TRANSFORMS
# ============================================================================

def get_transform():
    return T.Compose([T.Resize((INPUT_SIZE, INPUT_SIZE)), T.ToTensor()])

class VOCLikeDataset(torch.utils.data.Dataset):
    def __init__(self, images_dir, labels_dir, transforms=None, class_map=None):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.transforms = transforms
        self.class_map = class_map or CLASS_MAP
        self.files = sorted([f for f in os.listdir(images_dir) if f.endswith((".jpg",".png"))])
        
    def __len__(self): 
        return len(self.files)
        
    def __getitem__(self, idx):
        img_file = self.files[idx]
        img = Image.open(os.path.join(self.images_dir, img_file)).convert("RGB")
        xml_path = os.path.join(self.labels_dir, img_file.rsplit('.',1)[0] + '.xml')
        
        tree = ET.parse(xml_path)
        boxes, labels = [], []
        
        for obj in tree.getroot().findall('object'):
            cls = obj.find('name').text.lower().replace(" ", "_")
            if cls not in self.class_map:
                print(f"Warning: Unknown class '{cls}' found in {xml_path}")
                continue
                
            labels.append(self.class_map[cls])
            b = obj.find('bndbox')
            boxes.append([
                float(b.find('xmin').text), 
                float(b.find('ymin').text),
                float(b.find('xmax').text), 
                float(b.find('ymax').text)
            ])
        
        if self.transforms:
            img = self.transforms(img)
            
        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.int64)
        }
        return img, target

# ============================================================================
# VALIDATION FUNCTION
# ============================================================================

def validate(model, loader, device):
    """
    Runs model on loader, returns:
      - losses dict with keys 'box','cls','dfl'
      - metrics dict with keys 'precision','recall','mAP50','mAP50-95'
      - lists all_t, all_p, all_s for PR curves
    """
    model.eval()
    mp50 = MeanAveragePrecision(iou_thresholds=[0.5]).to(device)
    mp_all = MeanAveragePrecision().to(device)
    sum_b = sum_c = sum_d = 0.0
    count = 0
    all_t, all_p, all_s = [], [], []

    def match_predictions(pred_boxes, pred_labels, pred_scores, gt_boxes, gt_labels, iou_thresh=0.5):
        matches = []
        if len(pred_boxes) == 0:
            # No predictions, all ground truth are false negatives
            for j in range(len(gt_boxes)):
                matches.append((0, gt_labels[j].item(), 0.0))
            return matches
            
        if len(gt_boxes) == 0:
            # No ground truth, all predictions are false positives
            for i in range(len(pred_boxes)):
                matches.append((pred_labels[i].item(), 0, pred_scores[i].item()))
            return matches
            
        ious = box_iou(pred_boxes, gt_boxes)
        gt_used = set()
        
        for i in range(len(pred_boxes)):
            score = pred_scores[i].item()
            label = pred_labels[i].item()
            
            if ious.numel() > 0:
                max_iou, gt_idx = ious[i].max(0)
                if max_iou >= iou_thresh and gt_idx.item() not in gt_used:
                    matches.append((label, gt_labels[gt_idx].item(), score))
                    gt_used.add(gt_idx.item())
                    continue
            matches.append((label, 0, score))
            
        # Add unmatched ground truth as false negatives
        for j in range(len(gt_boxes)):
            if j not in gt_used:
                matches.append((0, gt_labels[j].item(), 0.0))
        return matches

    with torch.no_grad():
        for imgs, targets in loader:
            imgs = [img.to(device) for img in imgs]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            # Get predictions
            outputs = model(imgs)
            
            # Get training losses
            model.train()
            loss_dict = model(imgs, targets)
            model.eval()
            
            # Convert tensors to Python floats immediately
            sum_b += float(loss_dict['loss_box_reg'].item())
            sum_c += float(loss_dict['loss_classifier'].item())
            sum_d += float(loss_dict.get('loss_objectness', 0.0)) + float(loss_dict.get('loss_rpn_box_reg', 0.0))
            count += 1
            
            # Process predictions for metrics
            for out, tgt in zip(outputs, targets):
                keep = out['scores'] > SCORE_THRESHOLD
                pred = {
                    'boxes': out['boxes'][keep], 
                    'scores': out['scores'][keep], 
                    'labels': out['labels'][keep]
                }
                gt = {'boxes': tgt['boxes'], 'labels': tgt['labels']}
                
                mp50.update([pred], [gt])
                mp_all.update([pred], [gt])
                
                pb, pl, ps = pred['boxes'].cpu(), pred['labels'].cpu(), pred['scores'].cpu()
                gb, gl = gt['boxes'].cpu(), gt['labels'].cpu()
                
                for p_label, g_label, p_score in match_predictions(pb, pl, ps, gb, gl, iou_thresh=IOU_THRESHOLD):
                    all_p.append(p_label)
                    all_t.append(g_label)
                    all_s.append(p_score)

    res50 = float(mp50.compute()['map'].item())
    res_all = float(mp_all.compute()['map'].item())
    
    valid_labels = list(range(1, NUM_CLASSES))
    precision = precision_score(all_t, all_p, labels=valid_labels, average='weighted', zero_division=0)
    recall = recall_score(all_t, all_p, labels=valid_labels, average='weighted', zero_division=0)
    
    losses = {'box': sum_b/count, 'cls': sum_c/count, 'dfl': sum_d/count}
    metrics = {'precision': precision, 'recall': recall, 'mAP50': res50, 'mAP50-95': res_all}
    
    return losses, metrics, all_t, all_p, all_s

# ============================================================================
# PLOTTING FUNCTIONS
# ============================================================================

def create_confusion_matrix(y_true, y_pred, save_path, normalize=False):
    """Create confusion matrix plot"""
    labels = list(range(1, NUM_CLASSES))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    
    if normalize:
        cm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        cm = np.nan_to_num(cm)  # Handle division by zero
        title = 'Confusion Matrix (Normalized)'
        fmt = '.2f'
        vmax = 1.0
    else:
        title = 'Confusion Matrix'
        fmt = 'd'
        vmax = None
    
    # Calculate figure size based on number of classes
    fig_size = max(8, len(CLASS_NAMES) * 0.8)
    fig, ax = plt.subplots(figsize=(fig_size, fig_size))
    
    im = ax.imshow(cm, cmap='Blues', vmax=vmax)
    
    # Add text annotations
    for (i, j), v in np.ndenumerate(cm):
        color = 'white' if (normalize and v > 0.5) or (not normalize and v > cm.max()/2) else 'black'
        if normalize:
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', color=color, fontsize=8)
        else:
            ax.text(j, i, f'{v}', ha='center', va='center', color=color, fontsize=8)
    
    ax.set_xticks(range(len(CLASS_NAMES)), labels=CLASS_NAMES, rotation=45, ha='right')
    ax.set_yticks(range(len(CLASS_NAMES)), labels=CLASS_NAMES)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title)
    
    plt.colorbar(im)
    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

def create_confidence_curves(all_t, all_p, all_s, plot_dir):
    """Create F1, Precision, and Recall confidence curves"""
    labels = list(range(1, NUM_CLASSES))
    thr = np.linspace(0, 1, 200)
    
    # F1 Curve
    f1_curves = {}
    f1_all = []
    
    yta, ypa, ysa = np.array(all_t), np.array(all_p), np.array(all_s)
    
    for t in thr:
        yp = np.where(ysa >= t, ypa, 0)
        f1_all.append(f1_score(yta, yp, labels=labels, average='weighted', zero_division=0))
        
        # Calculate per-class F1
        for i, label in enumerate(labels):
            if label not in f1_curves:
                f1_curves[label] = []
            f1_curves[label].append(f1_score((yta == label).astype(int), (yp == label).astype(int), zero_division=0))
    
    # Plot F1 curve
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Plot top classes (limit to avoid clutter)
    top_classes = min(5, len(CLASS_NAMES))
    for i, (label, class_name) in enumerate(zip(labels[:top_classes], CLASS_NAMES[:top_classes])):
        ax.plot(thr, f1_curves[label], label=class_name, alpha=0.7)
    
    best = int(np.nanargmax(f1_all))
    ax.plot(thr, f1_all, linewidth=3, label=f'All Classes {f1_all[best]:.2f}@{thr[best]:.2f}', color='black')
    
    ax.set_xlabel('Confidence')
    ax.set_ylabel('F1')
    ax.set_title(f'F1-Confidence Curve (Top {top_classes} Classes)')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    fig.tight_layout()
    fig.savefig(plot_dir/'F1_curve.png', dpi=300, bbox_inches='tight')
    plt.close(fig)
    
    # Precision and Recall curves
    for metric_name, metric_func in [('Precision', precision_score), ('Recall', recall_score)]:
        fig, ax = plt.subplots(figsize=(12, 8))
        
        # Per-class curves
        for cls, class_name in zip(labels, CLASS_NAMES):
            mask = (yta == cls).astype(int)
            scores = np.where(ypa == cls, ysa, 0.0)
            
            if len(np.unique(mask)) > 1:  # Only plot if class exists
                prec_ci, rec_ci, thr_ci = precision_recall_curve(mask, scores)
                if metric_name == 'Precision':
                    y_vals = prec_ci[:-1]
                else:
                    y_vals = rec_ci[:-1]
                x_vals = thr_ci
                area = np.trapz(y_vals, x_vals) if len(x_vals) > 1 else 0
                ax.plot(x_vals, y_vals, label=f"{class_name} {area:.3f}", alpha=0.7)
        
        # All classes curve
        metric_all = []
        for t in thr:
            yp_thresh = np.where(ysa >= t, ypa, 0)
            metric_all.append(metric_func(yta, yp_thresh, labels=labels, average='weighted', zero_division=0))
        
        best_idx = int(np.nanargmax(metric_all))
        area_all = np.trapz(metric_all, thr)
        ax.plot(thr, metric_all, linewidth=3, color='black',
                label=f"All Classes {area_all:.3f} at {thr[best_idx]:.3f}")
        
        ax.set_xlabel('Confidence')
        ax.set_ylabel(metric_name)
        ax.set_title(f'{metric_name}–Confidence Curve')
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        fig.tight_layout()
        fig.savefig(plot_dir/f'{metric_name[0]}_curve.png', dpi=300, bbox_inches='tight')
        plt.close(fig)

def create_pr_curve(all_t, all_p, all_s, save_path):
    """Create Precision-Recall curve"""
    labels = list(range(1, NUM_CLASSES))
    yta, ypa, ysa = np.array(all_t), np.array(all_p), np.array(all_s)
    
    fig, ax = plt.subplots(figsize=(12, 8))
    
    ap_scores = []
    for cls, class_name in zip(labels, CLASS_NAMES):
        mask = (yta == cls).astype(int)
        scores = np.where(ypa == cls, ysa, 0)
        
        if len(np.unique(mask)) > 1:  # Only plot if class exists
            prec, rec, _ = precision_recall_curve(mask, scores)
            ap = np.trapz(prec, rec) if len(rec) > 1 else 0
            ap_scores.append(ap)
            ax.plot(rec, prec, label=f"{class_name} AP={ap:.3f}", alpha=0.7)
        else:
            ap_scores.append(0)
    
    # Overall mAP
    overall_map = np.mean(ap_scores)
    ax.plot([], [], linewidth=3, color='black', label=f'mAP = {overall_map:.3f}')
    
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title('Precision-Recall Curve')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

def create_label_distribution(dataset_base, save_path):
    """Create label distribution bar chart"""
    ann = []
    labels_path = Path(f"{dataset_base}/labels_voc/train")
    
    if not labels_path.exists():
        print(f"Warning: Labels path {labels_path} does not exist")
        return
    
    for xml_file in labels_path.glob('*.xml'):
        try:
            tree = ET.parse(xml_file)
            for obj in tree.getroot().findall('object'):
                class_name = obj.find('name').text
                ann.append(class_name)
        except Exception as e:
            print(f"Error reading {xml_file}: {e}")
    
    if not ann:
        print("No annotations found")
        return
    
    counts = pd.Series(ann).value_counts()
    
    fig, ax = plt.subplots(figsize=(12, 8))
    bars = counts.plot.bar(ax=ax, color='skyblue', edgecolor='black')
    ax.set_ylabel('Number of Instances')
    ax.set_xlabel('Class')
    ax.set_title('Label Distribution in Training Set')
    ax.tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for bar in bars.patches:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                f'{int(height)}', ha='center', va='bottom')
    
    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

def create_results_panel(csv_file, save_path):
    """Create training results panel"""
    if not os.path.exists(csv_file):
        print(f"Results CSV file {csv_file} not found")
        return
        
    df = pd.read_csv(csv_file)
    smooth = lambda x: uf(x.astype(float), size=5, mode='nearest')
    
    cols = [
        ('train/box_loss', 'Box Loss'),
        ('train/cls_loss', 'Cls Loss'),
        ('train/dfl_loss', 'DFL Loss'),
        ('metrics/precision(B)', 'Precision'),
        ('metrics/recall(B)', 'Recall'),
        ('metrics/mAP50(B)', 'mAP50'),
        ('metrics/mAP50-95(B)', 'mAP50-95'),
        ('val/box_loss', 'Val Box'),
        ('val/cls_loss', 'Val Cls'),
        ('val/dfl_loss', 'Val DFL')
    ]
    
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    axes = axes.flatten()
    
    for ax, (col, name) in zip(axes, cols):
        if col in df.columns:
            raw = df[col].astype(float)
            ax.plot(df['epoch'], raw, marker='o', label='raw', alpha=0.7, markersize=3)
            ax.plot(df['epoch'], smooth(raw), linestyle='--', label='smooth', linewidth=2)
            ax.set_title(name)
            ax.set_xlabel('Epoch')
            ax.legend()
            ax.grid(True, alpha=0.3)
        else:
            ax.text(0.5, 0.5, f'No data for\n{name}', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(name)
    
    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

# ============================================================================
# MAIN TRAINING FUNCTION
# ============================================================================

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Create datasets
    train_ds = VOCLikeDataset(f"{DATASET_BASE}/images/train", f"{DATASET_BASE}/labels_voc/train", get_transform())
    val_ds = VOCLikeDataset(f"{DATASET_BASE}/images/val", f"{DATASET_BASE}/labels_voc/val", get_transform())
    
    # Create data loaders
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
    val_loader = DataLoader(val_ds, batch_size=VAL_BATCH_SIZE, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))
    
    print(f"Training samples: {len(train_ds)}")
    print(f"Validation samples: {len(val_ds)}")
    
    # Create model
    model = fasterrcnn_resnet50_fpn(weights='DEFAULT')
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
    model.to(device)
    
    # Create optimizer and scheduler
    optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    
    # Prepare save directory and CSV
    save_dir = Path(f'runs/train/Faster_R-CNN_Optimized')
    save_dir.mkdir(parents=True, exist_ok=True)
    
    csv_file = save_dir / 'results.csv'
    headers = [
        'epoch', 'time', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss',
        'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)',
        'val/box_loss', 'val/cls_loss', 'val/dfl_loss', 'lr/pg0', 'lr/pg1', 'lr/pg2'
    ]
    
    best_map = 0
    patience_counter = 0

    # Training loop
    with open(csv_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        
        for epoch in range(1, NUM_EPOCHS + 1):
            model.train()
            t0 = time.time()
        
            if epoch <= WARMUP_EPOCH:
                warmup_lr = LEARNING_RATE * (epoch / WARMUP_EPOCH)
                for param_group in optimizer.param_groups:
                    param_group['lr'] = warmup_lr
                    param_group['momentum'] = 0.8
            
            # Training phase
            sum_box = sum_cls = sum_dfl = 0.0
            train_pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS}')
            
            for imgs, targets in train_pbar:
                imgs = [img.to(device) for img in imgs]
                targets = [{k: v.to(device) for k, v in target.items()} for target in targets]
                
                loss_dict = model(imgs, targets)
                losses = sum(loss_dict.values())
                
                sum_box += float(loss_dict['loss_box_reg'].item())
                sum_cls += float(loss_dict['loss_classifier'].item())
                sum_dfl += float(loss_dict.get('loss_objectness', 0.0)) + float(loss_dict.get('loss_rpn_box_reg', 0.0))
                
                optimizer.zero_grad()
                losses.backward()
                optimizer.step()
                
                # Update progress bar
                train_pbar.set_postfix({
                    'box': f'{sum_box/(train_pbar.n+1):.3f}',
                    'cls': f'{sum_cls/(train_pbar.n+1):.3f}'
                })
            
            # Calculate average training losses
            train_box = sum_box / len(train_loader)
            train_cls = sum_cls / len(train_loader)
            train_dfl = sum_dfl / len(train_loader)
            
            epoch_time = time.time() - t0
            
            # Validation phase
            print("Running validation...")
            val_losses, metrics, all_t, all_p, all_s = validate(model, val_loader, device)
            
            # Learning rates
            lrs = [group['lr'] for group in optimizer.param_groups]
            pg0, pg1, pg2 = (lrs + [lrs[-1]] * 3)[:3]
            
            # Write results
            row = [
                epoch, epoch_time,
                train_box, train_cls, train_dfl,
                metrics['precision'], metrics['recall'], metrics['mAP50'], metrics['mAP50-95'],
                val_losses['box'], val_losses['cls'], val_losses['dfl'],
                pg0, pg1, pg2
            ]
            writer.writerow(row)
            
            # Print epoch summary
            print(f"Epoch {epoch}/{NUM_EPOCHS} - Time: {epoch_time:.1f}s")
            print(f"  Train - Box: {train_box:.4f}, Cls: {train_cls:.4f}, DFL: {train_dfl:.4f}")
            print(f"  Val   - Box: {val_losses['box']:.4f}, Cls: {val_losses['cls']:.4f}, DFL: {val_losses['dfl']:.4f}")
            print(f"  Metrics - Precision: {metrics['precision']:.4f}, Recall: {metrics['recall']:.4f}")
            print(f"  mAP50: {metrics['mAP50']:.4f}, mAP50-95: {metrics['mAP50-95']:.4f}")
            print(f"  LR: {lrs[0]:.6f}")
            print("-" * 60)
            
            scheduler.step()
            
            # Early stopping check
            if metrics['mAP50'] > best_map:
                best_map = metrics['mAP50']
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter > PATIENCE:
                print(f"Early stopping at epoch {epoch} due to no improvement in mAP50")
                break
    
    # Final validation to collect predictions for plotting
    _, _, all_t, all_p, all_s = validate(model, val_loader, device)
    
    # Save predictions and model
    with open(save_dir / 'preds.pkl', 'wb') as f:
        pickle.dump((all_t, all_p, all_s), f)
    
    model_path = save_dir / 'model_final.pt'
    torch.save(model.state_dict(), model_path)
    print(f"Model weights saved to {model_path}")
    
    # Create plots
    print("Creating visualization plots...")
    plot_dir = save_dir / 'plots'
    plot_dir.mkdir(exist_ok=True)
    
    # 1. Confusion matrices
    create_confusion_matrix(all_t, all_p, plot_dir / 'confusion_matrix.png', normalize=False)
    create_confusion_matrix(all_t, all_p, plot_dir / 'confusion_matrix_normalized.png', normalize=True)
    
    # 2. Confidence curves (F1, Precision, Recall)
    create_confidence_curves(all_t, all_p, all_s, plot_dir)
    
    # 3. PR curve
    create_pr_curve(all_t, all_p, all_s, plot_dir / 'PR_curve.png')
    
    # 4. Label distribution
    create_label_distribution(DATASET_BASE, plot_dir / 'labels.png')
    
    # 5. Training results panel
    create_results_panel(csv_file, plot_dir / 'results.png')
    
    print(f"All plots saved under {plot_dir}")
    print(f"Training completed! Results saved in {save_dir}")

if __name__ == '__main__':
    main()

In [ ]:
import os
import shutil

import torch
import torchvision
import torchvision.transforms as T
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import DataLoader
from PIL import Image
import numpy as np
from sklearn.metrics import precision_score, recall_score, confusion_matrix, precision_recall_curve, f1_score
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torchvision.ops import box_iou
import xml.etree.ElementTree as ET
from tqdm import tqdm
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d as uf
import pickle
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader
from pathlib import Path
import pandas as pd
import torch

# Configuration
CLASS_NAMES = [
    "fall",
    "no_fall"
]

CLASS_MAP = {name.lower().replace(" ", "_"): idx + 1 for idx, name in enumerate(CLASS_NAMES)}
VISUAL_DEBUG = False
SCORE_THRESHOLD = 0.5
IOU_THRESHOLD = 0.5
INPUT_SIZE = 640

NUM_CLASSES = len(CLASS_NAMES) + 1
k = 5
DATASET_BASE = 'dataset_paper_new'
VAL_IMAGE_DIR = f"{DATASET_BASE}/images/train"
VAL_LABEL_DIR = f"{DATASET_BASE}/labels_voc/train"
TEMP_BASE = "temp_kfold"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 8
MODEL_PATH = "runs_2/train/FRCNN_noD_Reversed_lr.0.001//model_final.pt"

def get_transform():
    return T.Compose([T.Resize((INPUT_SIZE, INPUT_SIZE)), T.ToTensor()])

def validate(model, loader, device):
    """
    Runs model on loader, returns:
      - losses dict with keys 'box','cls','dfl'
      - metrics dict with keys 'precision','recall','mAP50','mAP50-95'
      - lists all_t, all_p, all_s for PR curves
    """
    model.eval()
    mp50 = MeanAveragePrecision(iou_thresholds=[0.5]).to(device)
    mp_all = MeanAveragePrecision().to(device)
    sum_b = sum_c = sum_d = 0.0
    count = 0
    all_t, all_p, all_s = [], [], []

    def match_predictions(pred_boxes, pred_labels, pred_scores, gt_boxes, gt_labels, iou_thresh=0.5):
        matches = []
        if len(pred_boxes) == 0:
            # No predictions, all ground truth are false negatives
            for j in range(len(gt_boxes)):
                matches.append((0, gt_labels[j].item(), 0.0))
            return matches
            
        if len(gt_boxes) == 0:
            # No ground truth, all predictions are false positives
            for i in range(len(pred_boxes)):
                matches.append((pred_labels[i].item(), 0, pred_scores[i].item()))
            return matches
            
        ious = box_iou(pred_boxes, gt_boxes)
        gt_used = set()
        
        for i in range(len(pred_boxes)):
            score = pred_scores[i].item()
            label = pred_labels[i].item()
            
            if ious.numel() > 0:
                max_iou, gt_idx = ious[i].max(0)
                if max_iou >= iou_thresh and gt_idx.item() not in gt_used:
                    matches.append((label, gt_labels[gt_idx].item(), score))
                    gt_used.add(gt_idx.item())
                    continue
            matches.append((label, 0, score))
            
        # Add unmatched ground truth as false negatives
        for j in range(len(gt_boxes)):
            if j not in gt_used:
                matches.append((0, gt_labels[j].item(), 0.0))
        return matches

    with torch.no_grad():
        for imgs, targets in loader:
            imgs = [img.to(device) for img in imgs]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            # Get predictions
            outputs = model(imgs)
            
            # Get training losses
            model.train()
            loss_dict = model(imgs, targets)
            model.eval()
            
            # Convert tensors to Python floats immediately
            sum_b += float(loss_dict['loss_box_reg'].item())
            sum_c += float(loss_dict['loss_classifier'].item())
            sum_d += float(loss_dict.get('loss_objectness', 0.0)) + float(loss_dict.get('loss_rpn_box_reg', 0.0))
            count += 1
            
            # Process predictions for metrics
            for out, tgt in zip(outputs, targets):
                keep = out['scores'] > SCORE_THRESHOLD
                pred = {
                    'boxes': out['boxes'][keep], 
                    'scores': out['scores'][keep], 
                    'labels': out['labels'][keep]
                }
                gt = {'boxes': tgt['boxes'], 'labels': tgt['labels']}
                
                mp50.update([pred], [gt])
                mp_all.update([pred], [gt])
                
                pb, pl, ps = pred['boxes'].cpu(), pred['labels'].cpu(), pred['scores'].cpu()
                gb, gl = gt['boxes'].cpu(), gt['labels'].cpu()
                
                for p_label, g_label, p_score in match_predictions(pb, pl, ps, gb, gl, iou_thresh=IOU_THRESHOLD):
                    all_p.append(p_label)
                    all_t.append(g_label)
                    all_s.append(p_score)

    res50 = float(mp50.compute()['map'].item())
    res_all = float(mp_all.compute()['map'].item())
    
    valid_labels = list(range(1, NUM_CLASSES))
    precision = precision_score(all_t, all_p, labels=valid_labels, average='weighted', zero_division=0)
    recall = recall_score(all_t, all_p, labels=valid_labels, average='weighted', zero_division=0)
    f1 = f1_score(all_t, all_p, labels=valid_labels, average='weighted', zero_division=0)
    accuracy = accuracy_score(all_t, all_p)

    losses = {
        'box': sum_b / count,
        'cls': sum_c / count,
        'dfl': sum_d / count
    }

    metrics = {
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'accuracy': accuracy,
        'mAP50': res50,
        'mAP50-95': res_all
    }

    return losses, metrics, all_t, all_p, all_s


class VOCLikeDataset(torch.utils.data.Dataset):
    def __init__(self, images_dir, labels_dir, transforms=None, class_map=None):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.transforms = transforms
        self.class_map = class_map or CLASS_MAP
        self.files = sorted([f for f in os.listdir(images_dir) if f.endswith((".jpg",".png"))])
        
    def __len__(self): 
        return len(self.files)
        
    def __getitem__(self, idx):
        img_file = self.files[idx]
        img = Image.open(os.path.join(self.images_dir, img_file)).convert("RGB")
        xml_path = os.path.join(self.labels_dir, img_file.rsplit('.',1)[0] + '.xml')
        
        tree = ET.parse(xml_path)
        boxes, labels = [], []
        
        for obj in tree.getroot().findall('object'):
            cls = obj.find('name').text.lower().replace(" ", "_")
            if cls not in self.class_map:
                print(f"Warning: Unknown class '{cls}' found in {xml_path}")
                continue
                
            labels.append(self.class_map[cls])
            b = obj.find('bndbox')
            boxes.append([
                float(b.find('xmin').text), 
                float(b.find('ymin').text),
                float(b.find('xmax').text), 
                float(b.find('ymax').text)
            ])
        
        if self.transforms:
            img = self.transforms(img)
            
        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.int64)
        }
        return img, target


os.makedirs(TEMP_BASE, exist_ok=True)

all_files = sorted([f for f in os.listdir(VAL_IMAGE_DIR) if f.lower().endswith((".jpg", ".png"))])
kfold = KFold(n_splits=k, shuffle=True, random_state=42)

model = fasterrcnn_resnet50_fpn(weights=None)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE).eval()

metrics_list = []

for fold, (_, val_idx) in enumerate(kfold.split(all_files), 1):
    print(f"\nRunning Fold {fold}...")
    
    fold_img_dir = os.path.join(TEMP_BASE, f"images_fold{fold}")
    fold_lbl_dir = os.path.join(TEMP_BASE, f"labels_fold{fold}")
    os.makedirs(fold_img_dir, exist_ok=True)
    os.makedirs(fold_lbl_dir, exist_ok=True)

    for i in val_idx:
        file = all_files[i]
        shutil.copy(os.path.join(VAL_IMAGE_DIR, file), fold_img_dir)
        xml_file = file.rsplit('.', 1)[0] + ".xml"
        shutil.copy(os.path.join(VAL_LABEL_DIR, xml_file), fold_lbl_dir)

    val_ds = VOCLikeDataset(fold_img_dir, fold_lbl_dir, get_transform())
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

    _, metrics, _, _, _ = validate(model, val_loader, DEVICE)
    print(f"Precision: {metrics['precision']:.4f} | Recall: {metrics['recall']:.4f} | F1 Score: {metrics['f1_score']:.4f} | Accuracy: {metrics['accuracy']:.4f} | mAP@0.5: {metrics['mAP50']:.4f} | mAP@0.5:0.95: {metrics['mAP50-95']:.4f}")
    metrics_list.append({
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "F1 Score": metrics["f1_score"],
        "Accuracy": metrics["accuracy"],
        "mAP50": metrics["mAP50"],
        "mAP50-95": metrics["mAP50-95"]
    })

shutil.rmtree(TEMP_BASE)

df = pd.DataFrame(metrics_list, index=[f"Fold {i}" for i in range(1, k+1)])
summary = df.agg(['mean', 'std']).rename(index={'mean': 'Mean', 'std': 'Std'})

print("\n=== Per-Fold Metrics ===")
print(df.to_string(float_format="%.4f"))
print("\n=== Cross-Validation Summary ===")
print(summary.to_string(float_format="%.4f"))


In [ ]:
#!/usr/bin/env python3
"""
Faster R-CNN Validation-Only Resource Monitor
---------------------------------------------
- NO TRAINING. Validates existing checkpoints for each LR in LEARNING_RATES.
- Expects weights under:
    runs_2/train/FRCNN_noD_Normal_lr.<lr> / weights file in WEIGHTS_PREFERENCE
  where WEIGHTS_PREFERENCE = ["model_final.pt", "best.pt", "last.pt"]

What it records per LR:
- per-batch inference time (ms), total validation wall time (s)
- process CPU%, process RAM (MB), system CPU%, system RAM%
- GPU utilization% & VRAM used (via NVML if available)
- PyTorch peak CUDA memory (allocated/reserved)
- mAP50, mAP50-95, precision, recall, and averaged val losses

Outputs:
- Per LR: runs_2/train/FRCNN_noD_Normal_lr.<lr>/val_monitor/{resource_summary.json, metrics.csv}
- Aggregate: runs_2/train/val_monitor_summary_FRCNN_valonly.csv

Prereqs:
  pip install torch torchvision psutil pynvml torchmetrics pandas tqdm pillow matplotlib scipy
"""

import os
import json
import time
import statistics as stats
from datetime import datetime
from pathlib import Path

import psutil
import torch
import torchvision.transforms as T
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import DataLoader
from PIL import Image
import numpy as np
from sklearn.metrics import precision_score, recall_score, confusion_matrix, precision_recall_curve, f1_score
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torchvision.ops import box_iou
import xml.etree.ElementTree as ET
from tqdm import tqdm
import pandas as pd

# Optional NVML for GPU telemetry
try:
    import pynvml
    _NVML_READY = True
    pynvml.nvmlInit()
except Exception:
    _NVML_READY = False

# ----------------------- Config (edit as needed) -----------------------------
CLASS_NAMES = ["fall", "no_fall"]
CLASS_MAP = {name.lower().replace(" ", "_"): idx + 1 for idx, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES) + 1  # + background
SCORE_THRESHOLD = 0.5
IOU_THRESHOLD = 0.5
INPUT_SIZE = 640

VAL_BATCH_SIZE = 1
DATASET_BASE = 'dataset_paper_new'

PROJECT_DIR = Path('runs_2/train')
NAME_PREFIX = 'FRCNN_noD_Reversed_lr'
LEARNING_RATES = [0.1, 0.01, 0.001]
WEIGHTS_PREFERENCE = ["model_final.pt", "best.pt", "last.pt"]  # tried in order

# ----------------------- Dataset --------------------------------------------
def get_transform():
    return T.Compose([T.Resize((INPUT_SIZE, INPUT_SIZE)), T.ToTensor()])

class VOCLikeDataset(torch.utils.data.Dataset):
    def __init__(self, images_dir, labels_dir, transforms=None, class_map=None):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.transforms = transforms
        self.class_map = class_map or CLASS_MAP
        self.files = sorted([f for f in os.listdir(images_dir) if f.lower().endswith((".jpg",".jpeg",".png"))])
    def __len__(self): 
        return len(self.files)
    def __getitem__(self, idx):
        img_file = self.files[idx]
        img = Image.open(os.path.join(self.images_dir, img_file)).convert("RGB")
        xml_path = os.path.join(self.labels_dir, img_file.rsplit('.',1)[0] + '.xml')
        boxes, labels = [], []
        try:
            tree = ET.parse(xml_path)
            for obj in tree.getroot().findall('object'):
                cls = obj.find('name').text.lower().replace(" ", "_")
                if cls not in self.class_map:
                    continue
                labels.append(self.class_map[cls])
                b = obj.find('bndbox')
                boxes.append([
                    float(b.find('xmin').text), 
                    float(b.find('ymin').text),
                    float(b.find('xmax').text), 
                    float(b.find('ymax').text)
                ])
        except Exception:
            pass
        if self.transforms:
            img = self.transforms(img)
        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.int64)
        }
        return img, target

# ----------------------- Monitoring utils -----------------------------------
_PROC = psutil.Process(os.getpid())

def _prime_cpu_percent_samplers():
    try: _PROC.cpu_percent(None)
    except Exception: pass
    try: psutil.cpu_percent(None)
    except Exception: pass

def get_cpu_ram_snapshot():
    try:
        p_cpu = _PROC.cpu_percent(None)
        rss_mb = _PROC.memory_info().rss / (1024**2)
    except Exception:
        p_cpu, rss_mb = None, None
    try:
        sys_cpu = psutil.cpu_percent(None)
        sys_mem = psutil.virtual_memory().percent
    except Exception:
        sys_cpu, sys_mem = None, None
    return p_cpu, rss_mb, sys_cpu, sys_mem

def update_gpu_maxima(max_gpu_util, max_gpu_mem_used):
    if not torch.cuda.is_available():
        return
    try:
        n = torch.cuda.device_count()
    except Exception:
        n = 0
    if n == 0:
        return
    if _NVML_READY:
        for i in range(n):
            try:
                h = pynvml.nvmlDeviceGetHandleByIndex(i)
                util = pynvml.nvmlDeviceGetUtilizationRates(h)
                mem = pynvml.nvmlDeviceGetMemoryInfo(h)
                max_gpu_util[i] = max(max_gpu_util.get(i, 0), int(util.gpu))
                used_mb = mem.used / (1024**2)
                max_gpu_mem_used[i] = max(max_gpu_mem_used.get(i, 0.0), used_mb)
            except Exception:
                pass
    else:
        for i in range(n):
            try:
                used_mb = torch.cuda.memory_allocated(i) / (1024**2)
                max_gpu_mem_used[i] = max(max_gpu_mem_used.get(i, 0.0), used_mb)
            except Exception:
                pass

class ValResourceMonitor:
    def __init__(self):
        self.batch_times = []
        self.max_proc_ram_mb = 0.0
        self.max_proc_cpu = 0.0
        self.max_sys_cpu = 0.0
        self.max_sys_mem = 0.0
        self.max_gpu_util = {}
        self.max_gpu_mem_used = {}
        self._t0 = None
        self._val_start = None
        self.summary = {}
    def start(self):
        _prime_cpu_percent_samplers()
        if torch.cuda.is_available():
            try: torch.cuda.reset_peak_memory_stats()
            except Exception: pass
        self._val_start = time.perf_counter()
        update_gpu_maxima(self.max_gpu_util, self.max_gpu_mem_used)
    def on_batch_start(self):
        if torch.cuda.is_available():
            try: torch.cuda.synchronize()
            except Exception: pass
        self._t0 = time.perf_counter()
    def on_batch_end(self):
        if torch.cuda.is_available():
            try: torch.cuda.synchronize()
            except Exception: pass
        if self._t0 is not None:
            self.batch_times.append(time.perf_counter() - self._t0)
        p_cpu, rss_mb, sys_cpu, sys_mem = get_cpu_ram_snapshot()
        if rss_mb is not None: self.max_proc_ram_mb = max(self.max_proc_ram_mb, rss_mb)
        if p_cpu is not None: self.max_proc_cpu = max(self.max_proc_cpu, p_cpu)
        if sys_cpu is not None: self.max_sys_cpu = max(self.max_sys_cpu, sys_cpu)
        if sys_mem is not None: self.max_sys_mem = max(self.max_sys_mem, sys_mem)
        update_gpu_maxima(self.max_gpu_util, self.max_gpu_mem_used)
    def finish(self):
        total_val_time = time.perf_counter() - (self._val_start or time.perf_counter())
        if torch.cuda.is_available():
            try:
                peak_alloc_mb = torch.cuda.max_memory_allocated() / (1024**2)
                peak_res_mb = torch.cuda.max_memory_reserved() / (1024**2)
            except Exception:
                peak_alloc_mb = peak_res_mb = None
        else:
            peak_alloc_mb = peak_res_mb = None
        if self.batch_times:
            bt_ms = [t * 1000.0 for t in self.batch_times]
            try: p50 = stats.median(bt_ms)
            except Exception: p50 = None
            try: p95 = stats.quantiles(bt_ms, n=20)[-1] if len(bt_ms) >= 2 else None
            except Exception: p95 = None
            avg = sum(bt_ms) / len(bt_ms)
        else:
            avg = p50 = p95 = None
        self.summary = {
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "num_batches": len(self.batch_times),
            "batch_time_ms_avg": avg,
            "batch_time_ms_p50": p50,
            "batch_time_ms_p95": p95,
            "total_val_time_s": total_val_time,
            "max_process_cpu_percent": self.max_proc_cpu,
            "max_process_ram_mb": self.max_proc_ram_mb,
            "max_system_cpu_percent": self.max_sys_cpu,
            "max_system_mem_percent": self.max_sys_mem,
            "max_gpu_util_percent": self.max_gpu_util,
            "max_gpu_mem_used_mb": self.max_gpu_mem_used,
            "torch_peak_cuda_alloc_mb": peak_alloc_mb,
            "torch_peak_cuda_reserved_mb": peak_res_mb,
            "nvml_available": _NVML_READY,
        }
        return self.summary

# ----------------------- Validation w/metrics --------------------------------
def validate_with_monitor(model, loader, device, save_dir, score_thresh=SCORE_THRESHOLD, iou_thresh=IOU_THRESHOLD):
    """
    Validation with metrics & resource monitoring.
    Writes: save_dir/val_monitor/{resource_summary.json, metrics.csv}
    """
    save_dir = Path(save_dir)
    vm_dir = save_dir / 'val_monitor'
    vm_dir.mkdir(parents=True, exist_ok=True)

    model.eval()
    mp50 = MeanAveragePrecision(iou_thresholds=[0.5]).to(device)
    mp_all = MeanAveragePrecision().to(device)
    sum_b = sum_c = sum_d = 0.0
    count = 0
    all_t, all_p, all_s = [], [], []

    def match_predictions(pred_boxes, pred_labels, pred_scores, gt_boxes, gt_labels, iou_thresh=0.5):
        matches = []
        if len(pred_boxes) == 0:
            for j in range(len(gt_boxes)):
                matches.append((0, gt_labels[j].item(), 0.0))
            return matches
        if len(gt_boxes) == 0:
            for i in range(len(pred_boxes)):
                matches.append((pred_labels[i].item(), 0, pred_scores[i].item()))
            return matches
        ious = box_iou(pred_boxes, gt_boxes)
        gt_used = set()
        for i in range(len(pred_boxes)):
            score = pred_scores[i].item()
            label = pred_labels[i].item()
            if ious.numel() > 0:
                max_iou, gt_idx = ious[i].max(0)
                if max_iou >= iou_thresh and gt_idx.item() not in gt_used:
                    matches.append((label, gt_labels[gt_idx].item(), score))
                    gt_used.add(gt_idx.item())
                    continue
            matches.append((label, 0, score))
        for j in range(len(gt_boxes)):
            if j not in gt_used:
                matches.append((0, gt_labels[j].item(), 0.0))
        return matches

    monitor = ValResourceMonitor()
    monitor.start()

    with torch.no_grad():
        for imgs, targets in loader:
            monitor.on_batch_start()

            imgs = [img.to(device) for img in imgs]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            # Predictions
            outputs = model(imgs)

            # Losses using training-style forward
            model.train()
            loss_dict = model(imgs, targets)
            model.eval()
            sum_b += float(loss_dict.get('loss_box_reg', 0.0))
            sum_c += float(loss_dict.get('loss_classifier', 0.0))
            sum_d += float(loss_dict.get('loss_objectness', 0.0)) + float(loss_dict.get('loss_rpn_box_reg', 0.0))
            count += 1

            # Metrics accumulation
            for out, tgt in zip(outputs, targets):
                keep = out['scores'] > score_thresh
                pred = {'boxes': out['boxes'][keep], 'scores': out['scores'][keep], 'labels': out['labels'][keep]}
                gt = {'boxes': tgt['boxes'], 'labels': tgt['labels']}
                mp50.update([pred], [gt])
                mp_all.update([pred], [gt])

                pb, pl, ps = pred['boxes'].detach().cpu(), pred['labels'].detach().cpu(), pred['scores'].detach().cpu()
                gb, gl = gt['boxes'].detach().cpu(), gt['labels'].detach().cpu()
                for p_label, g_label, p_score in match_predictions(pb, pl, ps, gb, gl, iou_thresh=iou_thresh):
                    all_p.append(p_label)
                    all_t.append(g_label)
                    all_s.append(p_score)

            monitor.on_batch_end()

    summary = monitor.finish()

    # Compute final metrics
    res50 = float(mp50.compute()['map'].item())
    res_all = float(mp_all.compute()['map'].item())
    valid_labels = list(range(1, NUM_CLASSES))
    precision = precision_score(all_t, all_p, labels=valid_labels, average='weighted', zero_division=0)
    recall = recall_score(all_t, all_p, labels=valid_labels, average='weighted', zero_division=0)
    losses = {'box': (sum_b/count) if count else 0.0, 'cls': (sum_c/count) if count else 0.0, 'dfl': (sum_d/count) if count else 0.0}
    metrics = {'precision': precision, 'recall': recall, 'mAP50': res50, 'mAP50-95': res_all}

    # Save outputs
    with open(vm_dir / 'resource_summary.json', 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2)
    pd.DataFrame([{**losses, **metrics}]).to_csv(vm_dir / 'metrics.csv', index=False)

    return losses, metrics, summary

# ----------------------- Loader for weights ----------------------------------
def build_model_and_load(weights_path, device):
    """Build Faster R-CNN model and load state dict or full module from weights_path."""
    model = fasterrcnn_resnet50_fpn(weights='DEFAULT')
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
    model.to(device)

    state = torch.load(weights_path, map_location=device)
    loaded = False

    # Try common patterns
    if isinstance(state, dict):
        # Case 1: pure state_dict
        try:
            model.load_state_dict(state)
            loaded = True
        except Exception:
            pass
        # Case 2: checkpoint-like
        if not loaded:
            for key in ('state_dict', 'model_state', 'model'):
                if key in state and isinstance(state[key], dict):
                    try:
                        model.load_state_dict(state[key])
                        loaded = True
                        break
                    except Exception:
                        pass

    if not loaded:
        raise RuntimeError(f"Could not load weights from {weights_path}. "
                           "Expected a state_dict or a checkpoint with 'state_dict'/'model_state' keys.")
    model.eval()
    return model

# ----------------------- Main -------------------------------------------------
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Data
    val_ds = VOCLikeDataset(f"{DATASET_BASE}/images/train", f"{DATASET_BASE}/labels_voc/train", get_transform())
    val_loader = DataLoader(val_ds, batch_size=VAL_BATCH_SIZE, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))
    print(f"Validation samples: {len(val_ds)}")

    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    agg_rows = []

    for lr in LEARNING_RATES:
        run_dir = PROJECT_DIR / f"{NAME_PREFIX}.{lr}"
        if not run_dir.exists():
            print(f"[SKIP] Run dir not found for lr={lr}: {run_dir}")
            continue

        # Find a weights file
        weights_path = None
        for fname in WEIGHTS_PREFERENCE:
            candidate = run_dir / fname
            if candidate.exists():
                weights_path = candidate
                break
        if weights_path is None:
            # Also check within 'weights' subdir in case user saved there
            for fname in WEIGHTS_PREFERENCE:
                candidate = run_dir / "weights" / fname
                if candidate.exists():
                    weights_path = candidate
                    break
        if weights_path is None:
            print(f"[SKIP] No weights found for lr={lr} in {run_dir} or weights/")
            continue

        print(f"\n=== Validating {weights_path} (lr={lr}) ===")
        model = build_model_and_load(weights_path, device)

        # Validate with monitoring
        _, metrics, summary = validate_with_monitor(model, val_loader, device, run_dir)

        row = {
            "lr0": lr,
            "weights": str(weights_path),
            "run_dir": str(run_dir),
            "total_val_time_s": summary.get("total_val_time_s"),
            "batch_time_ms_avg": summary.get("batch_time_ms_avg"),
            "batch_time_ms_p50": summary.get("batch_time_ms_p50"),
            "batch_time_ms_p95": summary.get("batch_time_ms_p95"),
            "max_process_cpu_percent": summary.get("max_process_cpu_percent"),
            "max_process_ram_mb": summary.get("max_process_ram_mb"),
            "max_system_cpu_percent": summary.get("max_system_cpu_percent"),
            "max_system_mem_percent": summary.get("max_system_mem_percent"),
            "max_gpu_util_percent": max(summary.get("max_gpu_util_percent", {}).values(), default=None) if isinstance(summary.get("max_gpu_util_percent"), dict) else None,
            "max_gpu_mem_used_mb": max(summary.get("max_gpu_mem_used_mb", {}).values(), default=None) if isinstance(summary.get("max_gpu_mem_used_mb"), dict) else None,
            "torch_peak_cuda_alloc_mb": summary.get("torch_peak_cuda_alloc_mb"),
            "torch_peak_cuda_reserved_mb": summary.get("torch_peak_cuda_reserved_mb"),
            "precision": metrics.get("precision"),
            "recall": metrics.get("recall"),
            "mAP50": metrics.get("mAP50"),
            "mAP50-95": metrics.get("mAP50-95"),
        }
        agg_rows.append(row)

    # Save aggregate CSV
    agg_path = PROJECT_DIR / 'val_monitor_summary_FRCNN_valonly_normal.csv'
    if agg_rows:
        pd.DataFrame(agg_rows).to_csv(agg_path, index=False)
        print(f"\nSaved aggregate summary: {agg_path.resolve()}")
    else:
        print("\nNo validations run. Check your run directories and weights.")

if __name__ == '__main__':
    main()
